# WF2 - Complete workflow: ParametricSimulation (ACCIM predefined model)

WF2 runs the complete predefined-model parametric workflow in a single notebook with explicit output and parameter control.

In practical terms, we will move through preparation, configuration, execution, and verification so every transition in the workflow stays explicit.
As the notebook progresses, each code cell is introduced with context first, then the call itself, and finally quick guidance on how to interpret the result.

Workflow roadmap:
- Preparation: imports, path configuration, and baseline model loading.
- Configuration: output selection, parameter definitions, and problem setup.
- Execution and review: simulation or optimization calls followed by targeted checks.

Main classes, methods, and functions used in this notebook:
- `AccimPredefModelsParamSim`
- `AccimPredefModelsParamSim.set_parameters`
- `AccimPredefModelsParamSim.run_parametric_simulation`
- `ef.get_building`
- `ParametricSimulation`
- `parametric.discover_available_outputs`
- `parametric.select_outputs`
- `parametric.apply_outputs_preflight`
- `parametric.set_outputs_for_simulation`
- `parametric.set_parameters`

Narrative structure used in this notebook:
- Each code cell is preceded by an explanatory markdown block.
- Section headings are contextual to the current notebook and execution block.
- Key arguments are described before they are used in code.
- Execution-oriented cells may include a follow-up markdown note to help interpret results.

## WF2 predefined parametric: import workflow dependencies

We start by loading every dependency required by the full workflow, so the remaining sections can run without hidden imports.

In this complete workflow notebook, imports are intentionally grouped in one place. That makes reruns in a fresh kernel predictable and easier to debug when versions change.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [ ]:
from pathlib import Path

import pandas as pd
import accim
from besos import eppy_funcs as ef
from accim.parametric_and_optimisation.main import ParametricSimulation

pd.set_option("display.max_columns", 160)


## WF2 predefined parametric: set project paths and discover IDF/EPW files

Here we declare the project assets (IDFs, EPWs, and results folder) used by the workflow.

For WF2 predefined parametric, this is the baseline configuration point. If anything changes in file naming or location, update this block first before touching later cells.

Variables created in this cell:
- `PROJECT_ROOT`: reused in subsequent workflow blocks.
- `idf_list`: reused in subsequent workflow blocks.
- `epw_list`: reused in subsequent workflow blocks.
- `OUTPUT_FREQ`: reused in subsequent workflow blocks.
- `RESULTS_DIRNAME`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `PROJECT_ROOT`, `idf_list`, `epw_list`, `OUTPUT_FREQ`.

In [ ]:
PROJECT_ROOT = Path.cwd()

idf_list = [
    "OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_high_performance.idf",
    "OSM_TestResidentialUnit_v01_onlygeometry_SchNatVent_v2520_low_performance.idf",
]
epw_list = ["Seville.epw", "Sydney.epw"]
OUTPUT_FREQ = "hourly"
RESULTS_DIRNAME = "nb12_main_parametric_predefined_results"

print("IDF files found:", [(PROJECT_ROOT / idf).exists() for idf in idf_list])
print("EPW files found:", [(PROJECT_ROOT / epw).exists() for epw in epw_list])


## WF2 predefined parametric: load the baseline IDF model

Load the base building model used by subsequent runs.

This is where the baseline IDF is loaded for all downstream operations. Later cells reuse this object for output setup, parameterization, and execution.

Key arguments used in this cell and why they matter:
- `idf_object`: defines this setting for the current call.
- `timesteps`: defines this setting for the current call.

Variables created in this cell:
- `building`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `building`.

In [ ]:
building = ef.get_building(idf_list[0])
accim.utils.reduce_runtime(idf_object=building, timesteps=2)


## WF2 predefined parametric: initialize parametric simulation object

At this point we instantiate the parametric engine that will coordinate outputs, parameters, and run plans.

From here onward, most workflow actions are methods on this object, which keeps the execution trace coherent.

Key arguments used in this cell and why they matter:
- `building`: loaded IDF model used in this workflow.
- `parameters_type`: parameter family used in the simulation setup.
- `epws`: list of EPW weather files used to run scenarios.
- `output_freqs`: time frequency requested for outputs.

Variables created in this cell:
- `parametric`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `parametric`.

In [ ]:
parametric = ParametricSimulation(
    building=building,
    parameters_type="accim predefined model",
    epws=epw_list,
    output_freqs=[OUTPUT_FREQ],
)


## WF2 predefined parametric: discover available meter and variable outputs

Before locking outputs, we scan what this model setup can actually report.

Doing this now prevents downstream failures caused by requesting unavailable variables or meter names.

Key arguments used in this cell and why they matter:
- `reduce_sim_time`: enables a fast scan mode when discovering outputs.
- `prefer`: preferred backend for output discovery.
- `meters`: energy meters to include as outputs.
- `match`: matching rule used to select outputs.
- `on_missing`: behavior when a requested output is missing.
- `suggest`: shows suggestions when no exact match is found.
- `variables`: output variables to include by name or pattern.

Variables created in this cell:
- `meter_names`: reused in subsequent workflow blocks.
- `auxiliary_variable_patterns`: reused in subsequent workflow blocks.
- `available_outputs`: reused in subsequent workflow blocks.
- `outputs_meta`: reused in subsequent workflow blocks.
- `selection_meters`: reused in subsequent workflow blocks.
- `df_meters_sel`: reused in subsequent workflow blocks.
- `selection_vars`: reused in subsequent workflow blocks.
- `df_vars_sel`: reused in subsequent workflow blocks.
- `missing_vars`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Check that the discovered meter/variable tables are not unexpectedly empty.
- Confirm the reported discovery source matches your preferred backend.

In [ ]:
meter_names = ["Heating:Electricity", "Cooling:Electricity"]
auxiliary_variable_patterns = [
    "Zone Operative Temperature",
    "Zone Thermal Comfort Fanger Model PMV",
    "Running Average Outdoor Air Temperature",
    "Setpoint Temperature_No Tolerance",
]
available_outputs = parametric.discover_available_outputs(
    reduce_sim_time=True,
    prefer="testsimeplus",
)
outputs_meta = available_outputs["meta"]
print("Discovery source:", outputs_meta.get("source"))

selection_meters = parametric.select_outputs(
    meters=meter_names,
    match="case_insensitive",
    on_missing="raise",
    suggest=True,
)
df_meters_sel = selection_meters["meters"]

selection_vars = parametric.select_outputs(
    variables=auxiliary_variable_patterns,
    match="contains",
    on_missing="warn",
    suggest=True,
)
df_vars_sel = selection_vars["variables"]
missing_vars = selection_vars.get("report", {}).get("missing", {}).get("variables", [])
print("Auxiliary hourly variables selected:", len(df_vars_sel))
if missing_vars:
    print("Missing auxiliary variables:", missing_vars)


## WF2 predefined parametric: run output preflight checks before execution

This block applies preflight checks to the selected outputs before execution.

Preflight is the final schema gate; if this step is clean, output-related runtime errors are much less likely.

Key arguments used in this cell and why they matter:
- `df_meters_sel`: table of selected meters to apply to the IDF.
- `df_vars_sel`: table of selected variables to apply to the IDF.
- `clean_mode`: cleanup mode applied before adding outputs.
- `validate_before_apply`: whether to validate before editing the IDF.
- `validate_after_apply`: whether to validate after editing the IDF.
- `on_missing`: behavior when a requested output is missing.

What to verify right after running this cell:
- Confirm the selected tables include all objective outputs you plan to track.
- Verify preflight did not report blocking missing outputs.

In [ ]:
parametric.apply_outputs_preflight(
    df_meters_sel=df_meters_sel,
    df_vars_sel=df_vars_sel,
    clean_mode="all",
    validate_before_apply=False,
    validate_after_apply=True,
    on_missing="raise",
)


## WF2 predefined parametric: apply selected output schema to simulation settings

Now we persist the validated output schema in the active workflow object.

After this point, runs use these exact output definitions, so this step should be revisited whenever target metrics change.

Key arguments used in this cell and why they matter:
- `df_output_meter`: final meter table used by the simulation.
- `df_output_variable`: final variable table used by the simulation.

What to verify right after running this cell:
- Confirm output schema assignment completed without warnings.
- Verify target meter/variable names match the intended analysis outputs.

In [ ]:
parametric.set_outputs_for_simulation(
    df_output_meter=df_meters_sel,
    df_output_variable=df_vars_sel,
)


## WF2 predefined parametric: define parameter ranges

This section defines the decision space and objective direction used in the run.

Treat this as the modeling core of the workflow: parameter ranges and minimization/maximization choices control result meaning.

Key arguments used in this cell and why they matter:
- `accis_params_dict`: ACCIS/aPMV parameter definition used in the study.

Variables created in this cell:
- `predef_parameters`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `predef_parameters`.

In [ ]:
predef_parameters = {
    "ComfStand": [1, 2],
    "HVACmode": [2],
    "CAT": [1, 2],
    "CATcoolOffset": [0],
    "CATheatOffset": [0],
}
parametric.set_parameters(accis_params_dict=predef_parameters)


## WF2 predefined parametric: configure problem definition

This section defines the decision space and objective direction used in the run.

Treat this as the modeling core of the workflow: parameter ranges and minimization/maximization choices control result meaning.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [ ]:
parametric.set_problem()


## WF2 predefined parametric: generate full predefined parameter combinations

This cell generates the full predefined parameter combination set for the workflow.

Generating the full set first gives a transparent baseline before adding IDF/EPW mapping columns in the next step.

What to verify right after running this cell:
- Review execution output and proceed when values look consistent with the workflow intent.

In [ ]:
parametric.sampling_full_set()


## WF2 predefined parametric: assemble the final execution dataframe

Here we assemble the final execution dataframe from sampled combinations and scenario labels.

This dataframe is the run contract: verify row structure now, because execution relies entirely on this table.

Variables created in this cell:
- `plan_df`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm these objects are available for the next block: `plan_df`.

In [ ]:
plan_df = parametric.parameters_values_df.copy()
plan_df["idf"] = [idf_list[i % len(idf_list)] for i in range(len(plan_df))]
print("Plan rows:", len(plan_df))
print(plan_df.head())


## WF2 predefined parametric: execute parametric run and inspect output rows

This is the execution checkpoint: we run the full parametric plan and collect output rows.

If row counts differ from expectations, inspect plan size and weather/model multipliers before continuing to analysis.

Key arguments used in this cell and why they matter:
- `out_dir`: output folder for generated artifacts and results.
- `df`: case plan dataframe executed in this run.
- `processes`: number of parallel processes for simulation runs.
- `keep_input`: whether generated input files are preserved.
- `keep_dirs`: whether per-run directories are preserved.

Variables created in this cell:
- `outputs`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Compare returned row count with the expected number of plan combinations.
- Inspect the first rows to ensure objective columns are present and populated.

In [ ]:
outputs = parametric.run_parametric_simulation(
    out_dir=RESULTS_DIRNAME,
    df=plan_df,
    processes=1,
    keep_input=True,
    keep_dirs=True,
)


### Reading the result

Review the row count and confirm it matches the expected combinations from your plan and EPW set.

## WF2 predefined parametric: preview execution outputs and row counts

We perform a quick sanity check on row counts and first records before post-processing.

This lightweight preview catches missing columns or unexpected schema changes early, while context is still fresh.

What to verify right after running this cell:
- Confirm row counts and first records look consistent with the execution settings.
- Check that expected objective/output columns are present before plotting or MCDM steps.

In [ ]:
print("Rows:", len(outputs))
print(outputs.head())


## WF2 predefined parametric: normalize datasets for cross-case comparison

In this stage we normalize outputs to make cross-case and cross-climate comparisons fairer.

Keep normalization choices consistent across workflows when preparing comparative figures or decision-support summaries.

Key arguments used in this cell and why they matter:
- `mode`: operation mode for the current method.
- `df_types`: dataset groups that should be normalized.

What to verify right after running this cell:
- Confirm normalization scope (`df_types`) matches the intended datasets.
- Check one or two normalized columns to verify value scaling behaves as expected.

In [ ]:
parametric.set_building_floor_area(mode="all")
parametric.normalize_outputs(df_types=["parametric"])


## WF2 predefined parametric: prepare plotting helper functions and display settings

This cell configures plotting helpers and deterministic figure-export behavior.

Keeping these utilities in one place simplifies the rest of the visualization section and avoids repeated plotting boilerplate.

Key arguments used in this cell and why they matter:
- `parents`: defines this setting for the current call.
- `exist_ok`: defines this setting for the current call.

Variables created in this cell:
- `plot_out_dir`: reused in subsequent workflow blocks.
- `norm_df`: reused in subsequent workflow blocks.
- `energy_cols`: reused in subsequent workflow blocks.
- `main_energy_col`: reused in subsequent workflow blocks.
- `secondary_energy_col`: reused in subsequent workflow blocks.
- `x_param`: reused in subsequent workflow blocks.
- `y_param`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Confirm helper functions are defined and plotting backend settings are correct for your session.
- Verify the plot output directory exists before generating figures.

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt
plot_out_dir = Path(f"{RESULTS_DIRNAME}/plots")
plot_out_dir.mkdir(parents=True, exist_ok=True)
norm_df = parametric.outputs_param_simulation.copy()
energy_cols = [c for c in norm_df.columns if "Heating:Electricity" in c or "Cooling:Electricity" in c]
main_energy_col = next(c for c in energy_cols if "Heating:Electricity" in c)
secondary_energy_col = next(c for c in energy_cols if "Cooling:Electricity" in c)
x_param = "ComfStand"
y_param = "CAT"


## WF2 predefined parametric: generate categorical boxplots for key metrics

We start the visualization stage with categorical boxplots to compare distribution spread across key parameter groupings.

This view is useful for quickly spotting median shifts and variability differences between climates or control settings.

Key arguments used in this cell and why they matter:
- `df_source`: defines this setting for the current call.
- `y_vars`: list of output variables to plot.
- `col`: facet column used in plotting.
- `row`: defines this setting for the current call.
- `hue`: field used to color plot traces.
- `out_dir`: output folder for generated artifacts and results.
- `normalize_per_m2`: whether results are normalized by floor area.
- `show_points`: whether individual points are overlaid on summary plots.
- `sharey`: whether facets share the same y-axis scale.
- `height`: height of each plotting panel.
- `aspect`: aspect ratio of each plotting panel.

Variables created in this cell:
- `fig_boxplots`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
# ACCIM 0.7.8: evitar hue="epw" en boxplots para no duplicar id_vars.
fig_boxplots = parametric.plot_categorical_boxplots(
    df_source="parametric",
    y_vars=[main_energy_col, secondary_energy_col],
    col=x_param,
    row=y_param,
    hue=None,
    out_dir=str(plot_out_dir),
    normalize_per_m2=False,
    show_points=True,
    sharey=True,
    height=3.6,
    aspect=1.1,
)
display(fig_boxplots.fig)
plt.show()


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

## WF2 predefined parametric: generate parametric scatter relationships

This cell builds scatter relationships between parameters and key outputs.

Scatter views help detect monotonic trends, nonlinear regions, and potential interaction patterns worth deeper analysis.

Key arguments used in this cell and why they matter:
- `x`: x-axis variable used in plotting.
- `y`: y-axis variable used in plotting.
- `df_source`: defines this setting for the current call.
- `hue`: field used to color plot traces.
- `style`: field used to style plot traces.
- `out_dir`: output folder for generated artifacts and results.
- `normalize_per_m2`: whether results are normalized by floor area.

Variables created in this cell:
- `fig_scatter`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
fig_scatter = parametric.plot_parametric_scatter(
    x=x_param,
    y=main_energy_col,
    df_source="parametric",
    hue=y_param,
    style="epw",
    out_dir=str(plot_out_dir),
    normalize_per_m2=False,
)
display(fig_scatter.fig)
plt.show()


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

We now create line plots to inspect how outputs evolve along ordered parameter values.

Line trends make cross-climate and cross-model contrasts easier to read when the x-axis has clear ordering.

Key arguments used in this cell and why they matter:
- `x`: x-axis variable used in plotting.
- `y_vars`: list of output variables to plot.
- `df_source`: defines this setting for the current call.
- `hue`: field used to color plot traces.
- `col`: facet column used in plotting.
- `out_dir`: output folder for generated artifacts and results.
- `normalize_per_m2`: whether results are normalized by floor area.
- `filename`: filename used when saving a figure.

Variables created in this cell:
- `fig_lines`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
fig_lines = parametric.plot_parametric_lines(
    x=y_param,
    y_vars=[main_energy_col, secondary_energy_col],
    df_source="parametric",
    hue="epw",
    col="idf",
    out_dir=str(plot_out_dir),
    normalize_per_m2=False,
)
for plot_path in fig_lines.values():
    display(Image(filename=str(plot_path)))


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

## WF2 predefined parametric: generate parametric heatmap views

Here we generate a heatmap representation of output response across a two-parameter grid.

Heatmaps are practical for identifying favorable parameter regions before narrowing the decision space.

Key arguments used in this cell and why they matter:
- `x`: x-axis variable used in plotting.
- `y`: y-axis variable used in plotting.
- `z`: z-axis or color-value variable used in plotting.
- `df_source`: defines this setting for the current call.
- `col`: facet column used in plotting.
- `out_dir`: output folder for generated artifacts and results.
- `normalize_per_m2`: whether results are normalized by floor area.
- `filename`: filename used when saving a figure.

Variables created in this cell:
- `fig_heatmap`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
fig_heatmap = parametric.plot_parametric_heatmap(
    x=x_param,
    y=y_param,
    z=main_energy_col,
    df_source="parametric",
    col="epw",
    out_dir=str(plot_out_dir),
    normalize_per_m2=False,
)
display(Image(filename=str(fig_heatmap)))


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

Here we build 2D density/hexbin plots to visualize concentration patterns in parameter-output space.

This view is useful for identifying dense performance regions and sparse outliers that may deserve targeted checks.

Key arguments used in this cell and why they matter:
- `x`: x-axis variable used in plotting.
- `y`: y-axis variable used in plotting.
- `kind`: plot kind (for example violin or hexbin).
- `df_source`: defines this setting for the current call.
- `col`: facet column used in plotting.
- `out_dir`: output folder for generated artifacts and results.
- `normalize_per_m2`: whether results are normalized by floor area.
- `filename`: filename used when saving a figure.

Variables created in this cell:
- `fig_density_2d`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
fig_density_2d = parametric.plot_parametric_density_2d(
    x=y_param,
    y=main_energy_col,
    kind="hexbin",
    df_source="parametric",
    col="epw",
    out_dir=str(plot_out_dir),
    normalize_per_m2=False,
)
display(Image(filename=str(fig_density_2d)))


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

In this section we generate the figures that support interpretation and reporting.

Each plot should be checked for labeling, grouping, and normalization assumptions before sharing results.

Key arguments used in this cell and why they matter:
- `x`: x-axis variable used in plotting.
- `y`: y-axis variable used in plotting.
- `z`: z-axis or color-value variable used in plotting.
- `df_source`: defines this setting for the current call.
- `col`: facet column used in plotting.
- `out_dir`: output folder for generated artifacts and results.
- `normalize_per_m2`: whether results are normalized by floor area.
- `filename`: filename used when saving a figure.

Variables created in this cell:
- `fig_contour`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
fig_contour = parametric.plot_parametric_contour(
    x=x_param,
    y=y_param,
    z=main_energy_col,
    df_source="parametric",
    col="epw",
    out_dir=str(plot_out_dir),
    normalize_per_m2=False,
)
display(Image(filename=str(fig_contour)))


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

This cell generates distribution plots for objective variables to compare spread and shape.

Distribution views complement Pareto plots by showing variability and overlap between parameter groups.

Key arguments used in this cell and why they matter:
- `x`: x-axis variable used in plotting.
- `y_vars`: list of output variables to plot.
- `kind`: plot kind (for example violin or hexbin).
- `df_source`: defines this setting for the current call.
- `hue`: field used to color plot traces.
- `out_dir`: output folder for generated artifacts and results.
- `normalize_per_m2`: whether results are normalized by floor area.
- `show_points`: whether individual points are overlaid on summary plots.
- `filename`: filename used when saving a figure.

Variables created in this cell:
- `fig_distributions`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
fig_distributions = parametric.plot_parametric_distributions(
    x=y_param,
    y_vars=[main_energy_col, secondary_energy_col],
    kind="violin",
    df_source="parametric",
    hue="epw",
    out_dir=str(plot_out_dir),
    normalize_per_m2=False,
    show_points=True,
)
for plot_path in fig_distributions.values():
    display(Image(filename=str(plot_path)))


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

## WF2 predefined parametric: generate ECDF views for distribution comparison

This block adds ECDF plots to compare output distributions across scenarios.

ECDF curves are especially useful when comparing the full distribution rather than only mean or median behavior.

Key arguments used in this cell and why they matter:
- `x`: x-axis variable used in plotting.
- `df_source`: defines this setting for the current call.
- `hue`: field used to color plot traces.
- `out_dir`: output folder for generated artifacts and results.
- `normalize_per_m2`: whether results are normalized by floor area.
- `filename`: filename used when saving a figure.

Variables created in this cell:
- `fig_ecdf`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
fig_ecdf = parametric.plot_parametric_ecdf(
    x=main_energy_col,
    df_source="parametric",
    hue="epw",
    out_dir=str(plot_out_dir),
    normalize_per_m2=False,
)
display(Image(filename=str(fig_ecdf)))


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

Here we build radar charts to compare grouped solutions across multiple metrics.

Radar views are helpful for communication when multiple performance dimensions must be interpreted simultaneously.

Key arguments used in this cell and why they matter:
- `metrics`: metrics included in the radar plot.
- `group_by`: field used to group traces in the radar plot.
- `df_source`: defines this setting for the current call.
- `out_dir`: output folder for generated artifacts and results.
- `normalize_per_m2`: whether results are normalized by floor area.
- `filename`: filename used when saving a figure.

Variables created in this cell:
- `fig_radar`: reused in subsequent workflow blocks.
- `radar_plot_path`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
fig_radar = parametric.plot_parametric_radar(
    metrics=[main_energy_col, secondary_energy_col, y_param, x_param],
    group_by="epw",
    df_source="parametric",
    out_dir=str(plot_out_dir),
    normalize_per_m2=False,
)
radar_plot_path = plot_out_dir / "plot_parametric_radar_parametric_group_epw.png"
display(Image(filename=str(radar_plot_path)))
display(fig_radar)


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

## WF2 predefined parametric: prepare hourly dataframe and plotting configuration

This block prepares hourly datasets and shared plotting settings used by the next hourly visualizations.

By centralizing hourly tokens, filters, and categorical ordering here, the following hourly plots remain concise and consistent.

Key arguments used in this cell and why they matter:
- `parents`: defines this setting for the current call.
- `exist_ok`: defines this setting for the current call.
- `output_columns`: output columns requested for hourly extraction.
- `file_source`: data source to read from (csv, parquet, etc.).
- `skip_confirmation`: skips interactive confirmation prompts.
- `start_date`: start timestamp used for time indexing.
- `normalize_per_m2`: whether results are normalized by floor area.
- `df_source`: defines this setting for the current call.
- `value_tokens`: defines this setting for the current call.
- `categorical_orders`: defines this setting for the current call.
- `row`: defines this setting for the current call.
- `col`: facet column used in plotting.

Variables created in this cell:
- `hourly_plot_dir`: reused in subsequent workflow blocks.
- `hourly_meter_tokens`: reused in subsequent workflow blocks.
- `hourly_aux_tokens`: reused in subsequent workflow blocks.
- `hourly_output_columns`: reused in subsequent workflow blocks.
- `hourly_value_tokens`: reused in subsequent workflow blocks.
- `parameter_orders`: reused in subsequent workflow blocks.
- `hourly_common`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
hourly_plot_dir = Path(plot_out_dir)
hourly_plot_dir.mkdir(parents=True, exist_ok=True)

hourly_meter_tokens = df_meters_sel["key_name"].dropna().astype(str).tolist()
hourly_aux_tokens = list(auxiliary_variable_patterns)
hourly_output_columns = list(dict.fromkeys(hourly_meter_tokens + hourly_aux_tokens))
hourly_value_tokens = hourly_meter_tokens

parameter_orders = {}
for p_col in [x_param, y_param]:
    if p_col in parametric.outputs_param_simulation.columns:
        parameter_orders[p_col] = sorted(pd.unique(parametric.outputs_param_simulation[p_col]).tolist())

parametric.get_hourly_df_parametric(
    output_columns=hourly_output_columns,
    file_source="csv",
    skip_confirmation=True,
    start_date="2024-01-01 01",
    normalize_per_m2=False,
)

hourly_common = dict(
    df_source="parametric_hourly",
    value_tokens=hourly_value_tokens,
    categorical_orders=parameter_orders,
    row=y_param,
    col=x_param,
    out_dir=str(hourly_plot_dir),
)

print("Hourly output columns requested:", hourly_output_columns)
print("Hourly value tokens:", hourly_value_tokens)
print("Hourly shape:", parametric.outputs_param_simulation_hourly.shape)


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

This block prepares hourly datasets and shared plotting settings used by the next hourly visualizations.

By centralizing hourly tokens, filters, and categorical ordering here, the following hourly plots remain concise and consistent.

Key arguments used in this cell and why they matter:
- `epw_filter`: EPW filter applied in hourly plots.
- `x`: x-axis variable used in plotting.
- `x_label`: x-axis label for exported figures.
- `y_label`: y-axis label for exported figures.
- `filename`: filename used when saving a figure.

Variables created in this cell:
- `scatter_grid`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
scatter_grid = parametric.plot_hourly_scatter(
    epw_filter="Seville",
    x="datetime",
    x_label="Time",
    y_label="Hourly Electricity [J]",
    filename="plot_hourly_scatter_time_seville.png",
    **hourly_common,
)
display(Image(filename=str(scatter_grid.saved_path)))


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.

## WF2 predefined parametric: prepare hourly dataframe and plotting configuration

This block prepares hourly datasets and shared plotting settings used by the next hourly visualizations.

By centralizing hourly tokens, filters, and categorical ordering here, the following hourly plots remain concise and consistent.

Key arguments used in this cell and why they matter:
- `epw_filter`: EPW filter applied in hourly plots.
- `y_label`: y-axis label for exported figures.
- `filename`: filename used when saving a figure.

Variables created in this cell:
- `line_grid`: reused in subsequent workflow blocks.

What to verify right after running this cell:
- Open exported figures and verify labels, grouping, and normalization settings.
- Confirm output files were saved in the expected folder.

In [ ]:
for city in ["Seville", "Sydney"]:
    line_grid = parametric.plot_hourly_lines(
        epw_filter=city,
        y_label="Hourly Electricity [J]",
        filename=f"plot_hourly_line_{city.lower()}.png",
        **hourly_common,
    )
    display(Image(filename=str(line_grid.saved_path)))


### Reading the result

Open saved figures and verify that filters, grouping, and labels match your intended interpretation.